# Cell 1: Imports & Preparation

In [32]:
%load_ext autoreload
%autoreload 2

import json
import os
import platform
import shutil
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import psutil

from digikala_review_transformer import data_io, labels, utils
from digikala_review_transformer.baseline import (
    DEFAULT_TOKEN_PATTERN,
    fit_and_cache,
    fit_eval_one,
    fit_final_candidates,
    load_cache,
    make_vectorizer,
    term_ranks,
    tokenize,
    top_ngrams,
    train_lr,
    tune_c,
    vocab_growth_probe,
)
from digikala_review_transformer.config import (
    ARTIFACTS,
    DATA_INTERIM,
    DATA_PROCESSED,
    PSEUDO_LABEL_THRESHOLD,
    REPORTS,
    ROOT,
    SPLIT_SEED,
)
from digikala_review_transformer.data_io import build_head_data, read_split
from digikala_review_transformer.error_analysis import body_overlap_share
from digikala_review_transformer.labels import (
    CLASSES,
    LABEL_COLS,
    class_weights,
    encode,
)
from digikala_review_transformer.metrics import (
    ab_comparison,
    band_by_rate_zero,
    build_metric_tables,
    dummy_predict,
    evaluate,
    evaluate_all,
    macro_f1_from_confusion,
)
from digikala_review_transformer.predictions import (
    read_predictions,
    register_model,
    write_predictions,
)
from digikala_review_transformer.utils import sha256_file, timer, write_json


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Cell 2: Parameters

In [33]:
# ---- Features / vectorizer ----
MODEL_TAG     = "tfidf12"
NGRAM         = (1, 2)
MIN_DF        = 5
MAX_FEATURES  = 500_000
SUBLINEAR_TF  = True
TOKEN_PATTERN = r"[\w\u200c]+"

# ---- Classifier / tuning ----
C_GRID       = [0.3, 1, 3, 10]
CLASS_WEIGHT = "balanced"
SOLVER, MAX_ITER, TOL = "lbfgs", 200, 1e-4
TUNE_N       = 500_000        # None = all rows
PROBE_N      = 200_000
N_JOBS       = 4

# ---- Analysis ----
HIGH_CONF    = 0.90
TOPK_NGRAMS  = 25

TOPN_MIN_DF = 100                      # second list: only features seen in >= this many train bodies
RANK_TERMS = ["معمولی", "بد نیست", "متوسط", "نیست", "نه", "عالی", "افتضاح"]

# ---- Expected values (from the masking phase) ----
EXPECTED_ROWS = {"train": 2_860_236, "validation": 922_794, "test": 922_805}
EXPECTED_REC_LABELED_TRAIN = 2_392_157
EXPECTED_BAND_MISSING = {"train": 0, "validation": 0, "test": 1}
MIN_FREE_DISK_GB = 10

# ---- Paths (derived from config roots) ----
ART_DIR    = ARTIFACTS / "baseline_tfidf"
REP_DIR    = REPORTS / "baseline_tfidf"
TABLES_DIR = REP_DIR / "tables"
FIG_DIR    = REP_DIR / "figures"
EXPLORER_DIR = REP_DIR / "explorer"
PRED_DIR   = REPORTS / "predictions"
CACHE_DIR  = DATA_INTERIM / "tfidf_cache"
FONT_DIR   = ROOT / "assets" / "fonts"
TIMING_CSV = TABLES_DIR / "timing.csv"
MANIFEST   = DATA_PROCESSED / "split_manifest.json"

PROBE_SIZES = [50_000, 100_000, 200_000, 300_000]  # nested random subsets for the vocabulary probe
MAX_PEAK_RAM_GB = 20       # budget for vectorizer fit (32 GB total, minus OS/kernel/data)
NEG_WORDS = ["نه", "نیست", "نبود"]  # negations that must survive tokenization

FULL_FIT_MAX_MIN = 5     # if one full fit is estimated under this, tune on all rows (TUNE_N=None)
TIE_TOL = 0.0005         # "tie" in validation Macro F1 -> smaller C wins
EXTEND_FACTOR = 3.0      # one extra C beyond the grid when the winner is on the edge
SLOWDOWN = 1.5           # parallel workers get fewer threads than the single probe fit

MAX_ITER_FINAL = 1000   # ceiling for final fits; lbfgs stops earlier when converged
FINAL_C_CANDIDATES = {"rec": [0.3, 1.0], "band": [0.1, 0.3]}  # both compared at convergence on full data
N_JOBS_FINAL = 8        # lbfgs is mostly single-threaded; 8 candidates run in parallel

EXT_C_CANDIDATES = {"rec": [0.1], "band": [0.03]}   # one-time extension below the lower edge
REGISTRY_CSV = REPORTS / "model_registry.csv"

for d in [ART_DIR, TABLES_DIR, FIG_DIR, EXPLORER_DIR, PRED_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Step 0 checks ----
import sklearn, scipy, joblib, pyarrow, PIL, arabic_reshaper, bidi  # import test
free_gb = shutil.disk_usage(ROOT).free / 1e9
fonts = sorted(p.name for p in FONT_DIR.glob("*.ttf")) if FONT_DIR.exists() else []
print(f"python {sys.version.split()[0]} | sklearn {sklearn.__version__} | scipy {scipy.__version__} | "
      f"joblib {joblib.__version__} | pyarrow {pyarrow.__version__} | pandas {pd.__version__}")
print(f"free disk: {free_gb:.1f} GB (min {MIN_FREE_DISK_GB}) | fonts: {fonts or 'none yet (needed from step 3)'}")
assert free_gb >= MIN_FREE_DISK_GB
print("STEP 0 PASSED")

python 3.12.16 | sklearn 1.9.1 | scipy 1.18.1 | joblib 1.6.0 | pyarrow 26.0.1 | pandas 3.1.1
free disk: 17.3 GB (min 10) | fonts: ['Poppins-Bold.ttf', 'Poppins-Medium.ttf', 'Poppins-Regular.ttf', 'Vazirmatn-Bold.ttf', 'Vazirmatn-Medium.ttf', 'Vazirmatn-Regular.ttf']
STEP 0 PASSED


# Cell 3: Step 1 (Loading and Control)

In [3]:
FILES = {n: DATA_PROCESSED / n for n in
         ["train_supervised.csv", "validation.csv", "test.csv",
          "train_pseudo_unlabeled.csv", "natural_unlabeled.csv"]}

with utils.timer("step1_read_csvs", TIMING_CSV):
    train = data_io.read_split(FILES["train_supervised.csv"])
    val   = data_io.read_split(FILES["validation.csv"])
    test  = data_io.read_split(FILES["test.csv"])
    pseudo_ids  = data_io.read_ids(FILES["train_pseudo_unlabeled.csv"])
    natural_ids = data_io.read_ids(FILES["natural_unlabeled.csv"])
splits = {"train": train, "validation": val, "test": test}
print(f"train in RAM: {train.memory_usage(deep=True).sum()/1e9:.2f} GB")

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
with utils.timer("step1_checks", TIMING_CSV):
    report = data_io.check_splits(
        splits, pseudo_ids, natural_ids, manifest, FILES,
        EXPECTED_ROWS, EXPECTED_REC_LABELED_TRAIN, EXPECTED_BAND_MISSING)
print(report.to_string(index=False))
report.to_csv(TABLES_DIR / "data_checks.csv", index=False)

# Label values and encoding sanity (raises if a value is unknown)
for head in ["rec", "band"]:
    for var in ["A", "B"]:
        col = labels.LABEL_COLS[(head, var)]
        d = data_io.build_head_data(train, head, var)
        print(f"{head}_{var} ({col}): train rows {len(d):,} | class codes {np.bincount(d['y']).tolist()}")
print("raw band values:", sorted(train[labels.RAW_COL['band']].dropna().unique()))
print("body empty/whitespace in train:", int((train['body'].str.strip() == '').sum()))

assert report["ok"].all(), "step 1 failed: stop and report the failed rows."
print("\nSTEP 1 PASSED")

[timer] step1_read_csvs: 6.0s
train in RAM: 0.60 GB
[timer] step1_checks: 35.6s
                                           check                                                              got                                                         expected   ok
                                     train: rows                                                          2860236                                                          2860236 True
                            train: duplicate ids                                                                0                                                                0 True
                   train: missing corrected band                                                                0                                                                0 True
     train: raw vs corrected band missing differ                                                                0                                                                0 True


# Cell 4: dummy baseline

In [6]:
# ---- Step 2: dummy baseline, class weights, overlap with train ----
import pandas as pd

with timer("step2_dummy_weights_overlap", TIMING_CSV):
    tr = read_split(DATA_PROCESSED / "train_supervised.csv")
    SPLITS = {"validation": read_split(DATA_PROCESSED / "validation.csv"),
              "test": read_split(DATA_PROCESSED / "test.csv")}

    # --- dummy: majority class of each variant's own TRAIN labels ---
    rows = []
    for head in ["rec", "band"]:
        for variant in ["A", "B"]:
            ytr = encode(tr[LABEL_COLS[(head, variant)]], head).dropna().astype(int)
            for split, df in SPLITS.items():
                raw = encode(df[LABEL_COLS[(head, "A")]], head)
                cor = encode(df[LABEL_COLS[(head, "B")]], head)
                code, pred = dummy_predict(ytr, len(df))
                res = evaluate_all(raw, cor, pred, head)
                res.insert(0, "model_id", f"dummy_{head}_{variant}")
                res.insert(1, "head", head)
                res.insert(2, "variant", variant)
                res.insert(3, "split", split)
                res.insert(4, "majority_class", CLASSES[head][code])
                rows.append(res)
    dummy = pd.concat(rows, ignore_index=True)
    dummy.to_csv(TABLES_DIR / "dummy_baseline.csv", index=False)

    # --- class weights (balanced: n / (k * n_c)) on train_supervised ---
    cw = {}
    for head in ["rec", "band"]:
        for variant in ["A", "B"]:
            y = encode(tr[LABEL_COLS[(head, variant)]], head).dropna().astype(int)
            w = class_weights(y.to_numpy(), 3)
            cw[f"{head}_{variant}"] = {
                "n_rows": int(len(y)),
                "weights": {CLASSES[head][c]: float(w[c]) for c in range(3)},
                "codes": {CLASSES[head][c]: c for c in range(3)},
            }
    write_json(cw, ART_DIR / "class_weights.json")

    # --- overlap with train (exact body match), by corrected rec label ---
    ov = body_overlap_share(tr["body"], SPLITS, LABEL_COLS[("rec", "B")])
    ov.to_csv(TABLES_DIR / "overlap_with_train.csv", index=False)

show = ["model_id", "split", "label_view", "n", "accuracy", "macro_f1", "balanced_accuracy"]
print(dummy[dummy["variant"] == "B"][show].round(4).to_string(index=False))
print("\nmajority classes:", dummy.groupby("model_id")["majority_class"].first().to_dict())
for k, v in cw.items():
    print(k, v["n_rows"], {a: round(b, 4) for a, b in v["weights"].items()})
print("\n", ov.round(2).to_string(index=False))

# sanity: values should match the expected dummy numbers
d = dummy[(dummy.variant == "B") & (dummy.split == "validation") & (dummy.label_view == "corrected")].set_index("head")
assert abs(d.loc["rec", "accuracy"] - 0.79) < 0.02 and abs(d.loc["rec", "macro_f1"] - 0.295) < 0.02
assert abs(d.loc["band", "accuracy"] - 0.66) < 0.02 and abs(d.loc["band", "macro_f1"] - 0.26) < 0.02
assert (abs(dummy["balanced_accuracy"] - 1/3) < 1e-9).all()
print("\nSTEP 2 PASSED")

[timer] step2_dummy_weights_overlap: 37.6s
    model_id      split label_view      n  accuracy  macro_f1  balanced_accuracy
 dummy_rec_B validation        raw 788677    0.7896    0.2941             0.3333
 dummy_rec_B validation  corrected 788677    0.7926    0.2948             0.3333
 dummy_rec_B       test        raw 787845    0.7891    0.2940             0.3333
 dummy_rec_B       test  corrected 787845    0.7922    0.2947             0.3333
dummy_band_B validation        raw 922794    0.6492    0.2624             0.3333
dummy_band_B validation  corrected 922794    0.6585    0.2647             0.3333
dummy_band_B       test        raw 922804    0.6489    0.2623             0.3333
dummy_band_B       test  corrected 922804    0.6581    0.2646             0.3333

majority classes: {'dummy_band_A': 'high', 'dummy_band_B': 'high', 'dummy_rec_A': 'recommended', 'dummy_rec_B': 'recommended'}
rec_A 2392157 {'recommended': 0.4351, 'not_recommended': 2.9885, 'no_idea': 2.7237}
rec_B 2392157 {'

# cell 5: Step 4 - tokenizer check

In [9]:
# ---- Step 4: tokenizer check (default sklearn pattern vs ours) ----
rng = np.random.default_rng(SPLIT_SEED)
sample30 = tr["body"].iloc[rng.choice(len(tr), 30, replace=False)].tolist()

diff_shown = 0
for b in sample30:
    a, o = tokenize(b, DEFAULT_TOKEN_PATTERN), tokenize(b, TOKEN_PATTERN)
    if a != o and diff_shown < 6:
        diff_shown += 1
        print("BODY   :", b[:160])
        print("DEFAULT:", a[:25])
        print("OURS   :", o[:25], "\n")

# statistics on a larger random sample
big = tr["body"].iloc[rng.choice(len(tr), 50_000, replace=False)].tolist()
tok_d = [tokenize(b, DEFAULT_TOKEN_PATTERN) for b in big]
tok_o = [tokenize(b, TOKEN_PATTERN) for b in big]
flat_d = [t for ts in tok_d for t in ts]
flat_o = [t for ts in tok_o for t in ts]
Z = "\u200c"
stats = {
    "tokens/body default": len(flat_d) / len(big),
    "tokens/body ours": len(flat_o) / len(big),
    "ZWNJ-containing tokens default": sum(Z in t for t in flat_d),
    "ZWNJ-containing tokens ours": sum(Z in t for t in flat_o),
    "single-char tokens ours (%)": 100 * sum(len(t) == 1 for t in flat_o) / len(flat_o),
    "ZWNJ-only or edge-ZWNJ tokens ours (%)": 100 * sum(t.startswith(Z) or t.endswith(Z) for t in flat_o) / len(flat_o),
    "bodies with ZWNJ (%)": 100 * np.mean([Z in b for b in big]),
}
for k, v in stats.items():
    print(f"{k}: {v:,.3f}")
for w in NEG_WORDS:
    cd, co = flat_d.count(w), flat_o.count(w)
    print(f"negation {w}: default={cd} ours={co}")
    assert cd == co, f"negation word {w} count differs"
assert stats["ZWNJ-containing tokens default"] == 0 and stats["ZWNJ-containing tokens ours"] > 0
print("\nSTEP 4 PASSED")

BODY   : من دنبال جا واسه پدهام بودم و خیلی خوبه برای روی میز آرایشی
DEFAULT: ['من', 'دنبال', 'جا', 'واسه', 'پدهام', 'بودم', 'خیلی', 'خوبه', 'برای', 'روی', 'میز', 'آرایشی']
OURS   : ['من', 'دنبال', 'جا', 'واسه', 'پدهام', 'بودم', 'و', 'خیلی', 'خوبه', 'برای', 'روی', 'میز', 'آرایشی'] 

BODY   : بنظرم خیلی خوبه وابسته به قیمتش و کلا ایرادی نداره شک نکنبد در خریدن
DEFAULT: ['بنظرم', 'خیلی', 'خوبه', 'وابسته', 'به', 'قیمتش', 'کلا', 'ایرادی', 'نداره', 'شک', 'نکنبد', 'در', 'خریدن']
OURS   : ['بنظرم', 'خیلی', 'خوبه', 'وابسته', 'به', 'قیمتش', 'و', 'کلا', 'ایرادی', 'نداره', 'شک', 'نکنبد', 'در', 'خریدن'] 

BODY   : راضی بودم نرم می‌کنه اما وزی و نمیگیره زیاد
DEFAULT: ['راضی', 'بودم', 'نرم', 'می', 'کنه', 'اما', 'وزی', 'نمیگیره', 'زیاد']
OURS   : ['راضی', 'بودم', 'نرم', 'می\u200cکنه', 'اما', 'وزی', 'و', 'نمیگیره', 'زیاد'] 

BODY   : خیلی خوشگل و خاصه جنسشم خوبه با دامن پلیسه عالی میشه من دوسش دارم
DEFAULT: ['خیلی', 'خوشگل', 'خاصه', 'جنسشم', 'خوبه', 'با', 'دامن', 'پلیسه', 'عالی', 'میشه', 'من', 'دوسش',

AssertionError: negation word نیست count differs

### Step 4 (revised negation check)

In [10]:
# ---- Step 4 (revised negation check): quantify the loss instead of requiring equality ----
from collections import Counter
MAX_NEG_LOSS_PCT = 5.0   # tolerated share of a negation word absorbed into longer ZWNJ-joined tokens

for w in NEG_WORDS:
    cd, co = flat_d.count(w), flat_o.count(w)
    lost = 100 * (cd - co) / max(cd, 1)
    longer = Counter(t for t in flat_o if w in t and t != w)
    print(f"negation {w}: default={cd} ours={co} lost={lost:.2f}% (lost = absorbed into longer tokens)")
    print("   longer tokens in ours:", [(t.replace(Z, "<ZWNJ>"), c) for t, c in longer.most_common(8)])
    assert co <= cd and lost <= MAX_NEG_LOSS_PCT, f"negation {w}: loss {lost:.2f}% above {MAX_NEG_LOSS_PCT}%"

assert stats["ZWNJ-containing tokens default"] == 0 and stats["ZWNJ-containing tokens ours"] > 0
print("\nSTEP 4 PASSED")

negation نه: default=680 ours=680 lost=0.00% (lost = absorbed into longer tokens)
   longer tokens in ours: [('پیشنهاد', 1635), ('میکنه', 1392), ('متاسفانه', 481), ('کنه', 417), ('نمیکنه', 370), ('هزینه', 196), ('اینه', 182), ('خونه', 181)]
negation نیست: default=3253 ours=3244 lost=0.28% (lost = absorbed into longer tokens)
   longer tokens in ours: [('نیستم', 156), ('نیستن', 61), ('بدنیست', 30), ('نیستش', 29), ('نیستند', 20), ('نیستید', 7), ('نیستا', 3), ('نیستین', 3)]
negation نبود: default=1403 ours=1398 lost=0.36% (lost = absorbed into longer tokens)
   longer tokens in ours: [('نبودم', 135), ('نبودن', 33), ('نبوده', 17), ('بدنبود', 12), ('نبودیم', 5), ('نبودند', 4), ('نبودش', 3), ('نبودنش', 2)]

STEP 4 PASSED


# Step 5

## Step 5a: Probe vocabulary size and RAM

In [11]:
# ---- Step 5a: vocabulary growth / RAM probe ----
with timer("step5a_probe", TIMING_CSV, sizes=str(PROBE_SIZES)):
    probe = vocab_growth_probe(tr["body"], PROBE_SIZES, NGRAM, TOKEN_PATTERN, MIN_DF, SPLIT_SEED)
probe.to_csv(TABLES_DIR / "vocab_probe.csv", index=False)
print(probe.round(3).to_string(index=False))

def extrapolate(col, n_target, k=3):
    """Power-law fit log(y) = a + b*log(n) on the largest k probe points."""
    x = np.log(probe["n_docs"].to_numpy()[-k:]); y = np.log(probe[col].to_numpy()[-k:])
    b, a = np.polyfit(x, y, 1)
    return float(np.exp(a + b * np.log(n_target))), float(b)

N_FULL = len(tr)
raw_pred, beta_raw = extrapolate("raw_vocab", N_FULL)
kept_pred, beta_kept = extrapolate("vocab_df_ge_min", N_FULL)
last = probe.iloc[-1]
bytes_per_entry = (last.rss_delta_gb - last.matrix_gb) * 1e9 / last.raw_vocab
vocab_ram = raw_pred * bytes_per_entry / 1e9
matrix_full = last.matrix_gb * N_FULL / last.n_docs
peak_est = 2 * (vocab_ram + matrix_full)            # factor 2: transient copies during fit
time_est_min = last.seconds / last.n_docs * N_FULL / 60

print(f"\ngrowth exponent raw={beta_raw:.3f} | df>={MIN_DF}={beta_kept:.3f}")
print(f"extrapolated raw vocab @ {N_FULL:,}: {raw_pred/1e6:.1f} M | with df>={MIN_DF}: {kept_pred/1e6:.2f} M")
print(f"bytes/entry ~ {bytes_per_entry:.0f} | vocab RAM ~ {vocab_ram:.1f} GB | matrix ~ {matrix_full:.1f} GB")
print(f"estimated peak RAM ~ {peak_est:.1f} GB (budget {MAX_PEAK_RAM_GB}) | estimated fit time ~ {time_est_min:.1f} min")
print("DECISION:", "FULL_FIT" if peak_est <= MAX_PEAK_RAM_GB else "SAMPLE_VOCAB (fit vocabulary on a seeded sample)")

[timer] step5a_probe: 9.9s
 n_docs  raw_vocab  vocab_df_ge_min     nnz  nnz_per_doc  seconds  rss_delta_gb  matrix_gb
  50000     234702            19888 1085769       21.715    0.748         0.045      0.009
 100000     393170            36065 2183268       21.833    1.438         0.077      0.018
 200000     652902            64418 4377419       21.887    2.822         0.131      0.036
 300000     870140            89951 6563597       21.879    4.153         0.172      0.054

growth exponent raw=0.724 | df>=5=0.832
extrapolated raw vocab @ 2,860,236: 4.5 M | with df>=5: 0.59 M
bytes/entry ~ 136 | vocab RAM ~ 0.6 GB | matrix ~ 0.5 GB
estimated peak RAM ~ 2.2 GB (budget 20) | estimated fit time ~ 0.7 min
DECISION: FULL_FIT


## cell 7: Step 5b: fit، transform, cache

In [13]:
# ---- Step 5b: fit vectorizer on train_supervised, transform val/test, cache ----
VEC_PATH = ART_DIR / f"vectorizer_{MODEL_TAG}.joblib"
BODIES = {"train": tr["body"], "validation": SPLITS["validation"]["body"], "test": SPLITS["test"]["body"]}
IDS = {"train": tr["id"].to_numpy(), "validation": SPLITS["validation"]["id"].to_numpy(),
       "test": SPLITS["test"]["id"].to_numpy()}

with timer("step5b_vectorizer_fit_transform_cache", TIMING_CSV, tag=MODEL_TAG):
    vec = make_vectorizer(NGRAM, TOKEN_PATTERN, MIN_DF, MAX_FEATURES, SUBLINEAR_TF)
    XS, meta = fit_and_cache(vec, BODIES, IDS, CACHE_DIR, MODEL_TAG, VEC_PATH)

print(f"vocab size: {meta['vocab_size']:,} (cap {MAX_FEATURES:,}) | pruned terms: {meta['pruned_terms']}")
print(f"min df among kept features: {meta['df_min_kept']} (MIN_DF={MIN_DF}) | median df: {meta['df_median_kept']:.0f}")
print(f"fit {meta['fit_seconds']:.0f}s | transform val+test {meta['transform_seconds']:.0f}s")
for k in XS:
    print(f"{k}: shape {meta['shape'][k]} | nnz {meta['nnz'][k]:,} ({meta['nnz_per_doc'][k]:.1f}/doc) | "
          f"zero rows {meta['zero_rows'][k]:,} | {meta['matrix_gb'][k]:.2f} GB | dtype {XS[k].dtype}")

voc = vec.vocabulary_
print("negations in vocab:", {w: (w in voc) for w in NEG_WORDS})
print("features with ZWNJ:", sum("\u200c" in f for f in voc), "| bigrams:", sum(" " in f for f in voc))
idf = vec.idf_
names = np.array(vec.get_feature_names_out())
print("lowest idf (most common):", list(names[np.argsort(idf)[:12]]))

# reload check: cache round-trip + reloaded vectorizer reproduces cached rows
XS2, _ = load_cache(CACHE_DIR, MODEL_TAG, IDS)
vec2 = joblib.load(VEC_PATH)
idx = np.random.default_rng(SPLIT_SEED).choice(len(IDS["validation"]), 1000, replace=False)
chk = vec2.transform(BODIES["validation"].iloc[idx])
diff = np.abs(chk - XS2["validation"][idx]).max()
print("max abs diff (reloaded vectorizer vs cache, 1000 val rows):", diff)

assert meta["vocab_size"] <= MAX_FEATURES and meta["df_min_kept"] >= MIN_DF
assert all(w in voc for w in NEG_WORDS)
assert diff < 1e-6
assert XS["train"].shape[0] == len(tr) == 2_860_236
print("\nSTEP 5b PASSED")

[timer] step5b_vectorizer_fit_transform_cache: 60.9s
vocab size: 500,000 (cap 500,000) | pruned terms: None
min df among kept features: 5 (MIN_DF=5) | median df: 11
fit 39s | transform val+test 17s
train: shape [2860236, 500000] | nnz 57,544,256 (20.1/doc) | zero rows 3,724 | 0.47 GB | dtype float32
validation: shape [922794, 500000] | nnz 15,467,273 (16.8/doc) | zero rows 1,033 | 0.13 GB | dtype float32
test: shape [922805, 500000] | nnz 15,423,463 (16.7/doc) | zero rows 1,050 | 0.13 GB | dtype float32
negations in vocab: {'نه': True, 'نیست': True, 'نبود': True}
features with ZWNJ: 7620 | bigrams: 447644
lowest idf (most common): ['و', 'خیلی', 'بود', 'به', 'از', 'خوب', 'خوبه', 'من', 'که', 'برای', 'ولی', 'با']
max abs diff (reloaded vectorizer vs cache, 1000 val rows): 0.0

STEP 5b PASSED


# Step 6: Probe timing and tuning the C value

## cell 8: Step 6a (probe)

In [16]:
# ---- Step 6a: timing probe (one fit, PROBE_N rows, C=1, rec_B) ----
HEADS = [("rec", "A"), ("rec", "B"), ("band", "A"), ("band", "B")]
MODEL_IDS = {h: f"{MODEL_TAG}_{h[0]}_{h[1]}" for h in HEADS}
TRAIN_PARAMS = dict(class_weight=CLASS_WEIGHT, solver=SOLVER, max_iter=MAX_ITER, tol=TOL)

DATA = {}
for head, var in HEADS:
    dtr = build_head_data(tr, head, var)
    dva = build_head_data(SPLITS["validation"], head, var)
    assert dtr["y"].notna().all() and dva["y"].notna().all()
    assert dtr["pos"].max() < XS["train"].shape[0] and dva["pos"].max() < XS["validation"].shape[0]
    DATA[MODEL_IDS[(head, var)]] = dict(
        head=head, variant=var,
        pos=dtr["pos"].to_numpy(), y=dtr["y"].astype("int64").to_numpy(),
        pos_val=dva["pos"].to_numpy(), y_val=dva["y"].astype("int64").to_numpy())
print({m: (len(d["pos"]), len(d["pos_val"])) for m, d in DATA.items()})

mid = MODEL_IDS[("rec", "B")]; d = DATA[mid]
sub = np.sort(np.random.default_rng(SPLIT_SEED).choice(len(d["pos"]), PROBE_N, replace=False))
with timer("step6a_probe_fit", TIMING_CSV, rows=PROBE_N, C=1):
    row, _ = fit_eval_one(mid, 1.0, XS["train"], d["pos"][sub], d["y"][sub],
                          XS["validation"], d["pos_val"], d["y_val"], "rec", TRAIN_PARAMS)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

sec_per_row = row["fit_seconds"] / PROBE_N
full_min = {m: sec_per_row * len(DATA[m]["pos"]) / 60 for m in DATA}
print("estimated full-fit minutes (linear in rows):", {m: round(v, 1) for m, v in full_min.items()})
TUNE_N_EFF = None if max(full_min.values()) <= FULL_FIT_MAX_MIN else TUNE_N
rows_used = [len(DATA[m]["pos"]) if TUNE_N_EFF is None else min(TUNE_N_EFF, len(DATA[m]["pos"])) for m in DATA]
est_wall = len(C_GRID) * sum(sec_per_row * r for r in rows_used) / 60 / N_JOBS * SLOWDOWN
print(f"TUNE_N_EFF = {TUNE_N_EFF} | estimated tuning wall time ~ {est_wall:.0f} min (grid only, no extension)")

{'tfidf12_rec_A': (2392157, 788677), 'tfidf12_rec_B': (2392157, 788677), 'tfidf12_band_A': (2860236, 922794), 'tfidf12_band_B': (2860236, 922794)}
[timer] step6a_probe_fit: 21.4s
{'model_id': 'tfidf12_rec_B', 'C': 1.0, 'val_macro_f1': 0.6732, 'val_accuracy': 0.8185, 'fit_seconds': 21.065, 'n_iter': 136, 'converged': True, 'n_train_rows': 200000, 'val_n': 788677}
estimated full-fit minutes (linear in rows): {'tfidf12_rec_A': 4.2, 'tfidf12_rec_B': 4.2, 'tfidf12_band_A': 5.0, 'tfidf12_band_B': 5.0}
TUNE_N_EFF = 500000 | estimated tuning wall time ~ 5 min (grid only, no extension)


## cell 9: Step 6b (tuning)

In [17]:
# ---- Step 6b: tune C per model on validation Macro F1 ----
with timer("step6b_tune_c", TIMING_CSV, tune_n=str(TUNE_N_EFF)):
    tuning, TUNED_MODELS = tune_c(XS["train"], XS["validation"], DATA, C_GRID, TRAIN_PARAMS,
                                  N_JOBS, TUNE_N_EFF, SPLIT_SEED, TIE_TOL, EXTEND_FACTOR)
cols = ["model_id", "C", "val_macro_f1", "val_accuracy", "fit_seconds", "n_iter", "converged",
        "n_train_rows", "tune_n", "val_n", "extended", "selected", "at_edge"]
tuning = tuning[cols].sort_values(["model_id", "C"]).reset_index(drop=True)
tuning.to_csv(TABLES_DIR / "tuning_results.csv", index=False)

print(tuning.round(4).to_string(index=False))
print("\nselected:\n", tuning[tuning.selected][["model_id", "C", "val_macro_f1", "val_accuracy", "n_iter", "converged", "at_edge"]]
      .round(4).to_string(index=False))
print("\nnot converged:", int((~tuning.converged).sum()), "of", len(tuning), "fits (max_iter =", MAX_ITER, ")")

assert tuning.groupby("model_id")["selected"].sum().eq(1).all()
floor = tuning["model_id"].str.contains("_rec_").map({True: 0.30, False: 0.27})
assert (tuning["val_macro_f1"] > floor).all(), "a fit is not above the dummy reference"
print("\nSTEP 6 PASSED")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  13 out of  16 | elapsed:  4.9min remaining:  1.1min
[Parallel(n_jobs=4)]: Done  16 out of  16 | elapsed:  6.0min finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


grid-edge winners, extending once: [('tfidf12_band_A', 0.1), ('tfidf12_band_B', 0.1)]
[timer] step6b_tune_c: 389.7s
      model_id    C  val_macro_f1  val_accuracy  fit_seconds  n_iter  converged  n_train_rows  tune_n  val_n  extended  selected  at_edge
tfidf12_band_A  0.1        0.5657        0.6544      26.1156     114       True        500000  500000 922794      True     False    False
tfidf12_band_A  0.3        0.5672        0.6596      75.3608     180       True        500000  500000 922794     False      True    False
tfidf12_band_A  1.0        0.5583        0.6546      93.2972     200      False        500000  500000 922794     False     False    False
tfidf12_band_A  3.0        0.5513        0.6551      95.2632     200      False        500000  500000 922794     False     False    False
tfidf12_band_A 10.0        0.5298        0.6258      93.9360     200      False        500000  500000 922794     False     False    False
tfidf12_band_B  0.1        0.5727        0.6626      27.

[Parallel(n_jobs=4)]: Done   2 out of   2 | elapsed:   27.9s finished


# Step 7: Final fit of the 4 models on all labeled training rows.

In [23]:
# ---- Step 7: final fit of candidates on all labeled train rows; pick C on validation ----
FINAL_PARAMS = {**TRAIN_PARAMS, "max_iter": MAX_ITER_FINAL}
CAND = {mid: FINAL_C_CANDIDATES[d["head"]] for mid, d in DATA.items()}
print("candidates:", CAND, "| max_iter:", MAX_ITER_FINAL, "| workers:", N_JOBS_FINAL)

with timer("step7_final_fit", TIMING_CSV, max_iter=MAX_ITER_FINAL, n_candidates=sum(map(len, CAND.values()))):
    final = fit_final_candidates(XS["train"], XS["validation"], DATA, CAND, FINAL_PARAMS,
                                 N_JOBS_FINAL, ART_DIR, TIE_TOL)

final["sha256"] = [sha256_file(p) if p else "" for p in final["artifact"]]
final["size_mb"] = [Path(p).stat().st_size / 1e6 if p else None for p in final["artifact"]]
final = final.sort_values(["model_id", "C"]).reset_index(drop=True)
final.to_csv(TABLES_DIR / "final_fit.csv", index=False)

cols = ["model_id", "C", "val_macro_f1", "val_macro_recall", "val_accuracy", "n_iter", "converged",
        "fit_seconds", "selected"]
print(final[cols].round(4).to_string(index=False))

# which C would the alternative criterion (Macro Recall) pick? informational only
alt = final.loc[final.groupby("model_id")["val_macro_recall"].idxmax(), ["model_id", "C", "val_macro_recall"]]
print("\nC by Macro F1 (used):", final[final.selected].set_index("model_id")["C"].to_dict())
print("C by Macro Recall (info):", alt.set_index("model_id")["C"].to_dict())

# compare with the tuning table (500k rows, 200 iterations) at the same C
for _, r in final[final.selected].iterrows():
    t = tuning[(tuning.model_id == r.model_id) & (tuning.C == r.C)]["val_macro_f1"]
    print(f"{r.model_id}: final macro_f1={r.val_macro_f1:.4f} | tuning(500k,200it)={t.iloc[0]:.4f}")

# reload check: saved winner reproduces the stored validation score
for _, r in final[final.selected].iterrows():
    d = DATA[r.model_id]
    m = joblib.load(r.artifact)
    assert list(m.classes_) == [0, 1, 2]
    s = evaluate(d["y_val"], m.predict(XS["validation"][d["pos_val"]]), 3)
    assert abs(s["macro_f1"] - r.val_macro_f1) < 1e-9, f"{r.model_id}: reloaded model differs"

selected = final[final.selected]
cset = {mid: sorted(c) for mid, c in CAND.items()}
edge = [(r.model_id, r.C) for _, r in selected.iterrows() if r.C in (cset[r.model_id][0], cset[r.model_id][-1])]
print("\nwinners at the edge of the candidate set (curve not closed):", edge)
assert selected["converged"].all(), "a selected fit did not converge: raise MAX_ITER_FINAL and log it"
print("\nSTEP 7 PASSED")

candidates: {'tfidf12_rec_A': [0.3, 1.0], 'tfidf12_rec_B': [0.3, 1.0], 'tfidf12_band_A': [0.1, 0.3], 'tfidf12_band_B': [0.1, 0.3]} | max_iter: 1000 | workers: 8


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   3 out of   8 | elapsed:  6.9min remaining: 11.6min
[Parallel(n_jobs=8)]: Done   5 out of   8 | elapsed:  7.5min remaining:  4.5min
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:  8.0min finished


[timer] step7_final_fit: 480.5s
      model_id   C  val_macro_f1  val_macro_recall  val_accuracy  n_iter  converged  fit_seconds  selected
tfidf12_band_A 0.1        0.5762            0.5867        0.6663     207       True     385.1835      True
tfidf12_band_A 0.3        0.5759            0.5857        0.6674     221       True     413.5740     False
tfidf12_band_B 0.1        0.5834            0.5980        0.6746     241       True     447.9840      True
tfidf12_band_B 0.3        0.5832            0.5965        0.6762     224       True     462.0996     False
 tfidf12_rec_A 0.3        0.6847            0.7374        0.8135     254       True     400.8309      True
 tfidf12_rec_A 1.0        0.6813            0.7291        0.8151     346       True     478.3993     False
 tfidf12_rec_B 0.3        0.6874            0.7432        0.8169     262       True     429.5699      True
 tfidf12_rec_B 1.0        0.6849            0.7364        0.8186     310       True     453.8785     False

C by

## cell 11: Step 7b (Extend)

In [27]:
# ---- Step 7b: one-time extension below the lower edge; selection over the union ----
prev_final = final.copy()
CAND_EXT = {mid: EXT_C_CANDIDATES[d["head"]] for mid, d in DATA.items()}
print("new candidates:", CAND_EXT)

with timer("step7b_extend_fit", TIMING_CSV, n_candidates=sum(map(len, CAND_EXT.values()))):
    final = fit_final_candidates(XS["train"], XS["validation"], DATA, CAND_EXT, FINAL_PARAMS,
                                 4, ART_DIR, TIE_TOL, prev=prev_final)

final["sha256"] = [sha256_file(p) if p else "" for p in final["artifact"]]
final["size_mb"] = [Path(p).stat().st_size / 1e6 if p else None for p in final["artifact"]]
final = final.sort_values(["model_id", "C"]).reset_index(drop=True)
final.to_csv(TABLES_DIR / "final_fit.csv", index=False)

cols = ["model_id", "C", "val_macro_f1", "val_macro_recall", "val_accuracy", "n_iter", "converged",
        "fit_seconds", "selected"]
print(final[cols].round(4).to_string(index=False))
print("\nC by Macro F1 (used):", final[final.selected].set_index("model_id")["C"].to_dict())
alt = final.loc[final.groupby("model_id")["val_macro_recall"].idxmax(), ["model_id", "C"]]
print("C by Macro Recall (info):", alt.set_index("model_id")["C"].to_dict())

lo = final.groupby("model_id")["C"].min(); hi = final.groupby("model_id")["C"].max()
edge = [(r.model_id, r.C) for _, r in final[final.selected].iterrows() if r.C in (lo[r.model_id], hi[r.model_id])]
print("winners still at the edge (no further extension, report only):", edge)

# reload check: saved winner reproduces the stored validation score
for _, r in final[final.selected].iterrows():
    d = DATA[r.model_id]
    m = joblib.load(r.artifact)
    assert list(m.classes_) == [0, 1, 2]
    s = evaluate(d["y_val"], m.predict(XS["validation"][d["pos_val"]]), 3)
    assert abs(s["macro_f1"] - r.val_macro_f1) < 1e-9, f"{r.model_id}: saved model differs from stored score"

assert final[final.selected]["converged"].all(), "a selected fit did not converge"
assert final.groupby("model_id")["selected"].sum().eq(1).all()
print("\nSTEP 7b PASSED")

new candidates: {'tfidf12_rec_A': [0.1], 'tfidf12_rec_B': [0.1], 'tfidf12_band_A': [0.03], 'tfidf12_band_B': [0.03]}


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:  2.3min remaining:  2.3min
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:  2.4min finished


[timer] step7b_extend_fit: 143.0s
      model_id    C  val_macro_f1  val_macro_recall  val_accuracy  n_iter  converged  fit_seconds  selected
tfidf12_band_A 0.03        0.5707            0.5836        0.6587     135       True     121.9571     False
tfidf12_band_A 0.10        0.5762            0.5867        0.6663     207       True     385.1835      True
tfidf12_band_A 0.30        0.5759            0.5857        0.6674     221       True     413.5740     False
tfidf12_band_B 0.03        0.5787            0.5949        0.6686     147       True     136.3603     False
tfidf12_band_B 0.10        0.5834            0.5980        0.6746     241       True     447.9840      True
tfidf12_band_B 0.30        0.5832            0.5965        0.6762     224       True     462.0996     False
 tfidf12_rec_A 0.10        0.6808            0.7373        0.8082     186       True     141.3503     False
 tfidf12_rec_A 0.30        0.6847            0.7374        0.8135     254       True     400.8309     

# cell 12: Step 8 (predictions , registry)

In [28]:
# ---- Step 8: predictions per model and split + model registry ----
sel = final[final.selected].set_index("model_id")
with timer("step8_predictions", TIMING_CSV):
    for mid, d in DATA.items():
        head, var = d["head"], d["variant"]
        r = sel.loc[mid]
        model = joblib.load(r["artifact"])
        assert list(model.classes_) == [0, 1, 2]
        for split, df in SPLITS.items():
            proba = model.predict_proba(XS[split])
            raw = encode(df[LABEL_COLS[(head, "A")]], head)
            cor = encode(df[LABEL_COLS[(head, "B")]], head)
            write_predictions(PRED_DIR, mid, split, df["id"].to_numpy(), raw, cor, proba, head)
        register_model(REGISTRY_CSV, dict(
            model_id=mid, family="tfidf_lr", head=head, variant=var,
            train_label_col=LABEL_COLS[(head, var)], ngram=str(NGRAM), C=float(r["C"]),
            class_weight=CLASS_WEIGHT, n_train_rows=len(d["pos"]),
            artifact_path=Path(os.path.relpath(r["artifact"], ROOT)).as_posix()))

# verification: id order == CSV order, no text columns, own-label validation score == stored score
chk = []
for mid, d in DATA.items():
    own = "y_true_corr" if d["variant"] == "B" else "y_true_raw"
    for split, df in SPLITS.items():
        p = read_predictions(PRED_DIR, mid, split)
        assert np.array_equal(p["id"].to_numpy(), df["id"].astype(str).to_numpy()), f"{mid}/{split}: id order"
        assert not ({"body", "product_id"} & set(p.columns))
        assert p["pred"].between(0, 2).all() and p["conf"].between(1 / 3 - 1e-3, 1).all()
        if split == "validation":
            m = evaluate(p[own], p["pred"], 3, ordinal=(d["head"] == "band"))
            assert abs(m["macro_f1"] - sel.loc[mid, "val_macro_f1"]) < 1e-9, f"{mid}: parquet score differs"
            chk.append({"model_id": mid, "val_n": m["n"], "own_label_macro_f1": round(m["macro_f1"], 4),
                        "mean_conf": round(float(p["conf"].mean()), 4)})
print(pd.DataFrame(chk).to_string(index=False))

files = sorted(PRED_DIR.glob(f"{MODEL_TAG}_*.parquet"))
print([f"{f.name} {f.stat().st_size/1e6:.0f}MB" for f in files])
reg = pd.read_csv(REGISTRY_CSV, keep_default_na=False)
print(reg.drop(columns=["artifact_path"]).to_string(index=False))
assert len(files) == 8 and reg["model_id"].str.startswith(MODEL_TAG).sum() == 4
assert all((ROOT / p).exists() for p in reg["artifact_path"])
print("\nSTEP 8 PASSED")

[timer] step8_predictions: 5.4s
      model_id  val_n  own_label_macro_f1  mean_conf
 tfidf12_rec_A 788677              0.6847     0.7896
 tfidf12_rec_B 788677              0.6874     0.7972
tfidf12_band_A 922794              0.5762     0.5951
tfidf12_band_B 922794              0.5834     0.6054
['tfidf12_band_A__test.parquet 24MB', 'tfidf12_band_A__validation.parquet 24MB', 'tfidf12_band_B__test.parquet 24MB', 'tfidf12_band_B__validation.parquet 24MB', 'tfidf12_rec_A__test.parquet 24MB', 'tfidf12_rec_A__validation.parquet 24MB', 'tfidf12_rec_B__test.parquet 24MB', 'tfidf12_rec_B__validation.parquet 24MB']
      model_id   family head variant       train_label_col  ngram   C class_weight  n_train_rows                created_at
 tfidf12_rec_A tfidf_lr  rec       A recommendation_status (1, 2) 0.3     balanced       2392157 2026-11-12T10:12:59+03:30
 tfidf12_rec_B tfidf_lr  rec       B       step1_rec_label (1, 2) 0.3     balanced       2392157 2026-11-12T10:13:00+03:30
tfidf12_band_A tf

# cell 13: Step 9

In [31]:
# ---- Step 9: metric tables computed from the saved prediction files ----
MODELS = {mid: (d["head"], d["variant"]) for mid, d in DATA.items()}
BAND_MODELS = {m: v for m, v in MODELS.items() if v[0] == "band"}
RATES = {s: SPLITS[s]["rate"].to_numpy() for s in SPLITS}

with timer("step9_metric_tables", TIMING_CSV):
    summary, per_class, conf = build_metric_tables(PRED_DIR, MODELS)
    ab = ab_comparison(summary, PRED_DIR, MODELS)
    rz = band_by_rate_zero(PRED_DIR, BAND_MODELS, RATES)

summary.to_csv(TABLES_DIR / "metrics_summary.csv", index=False)
per_class.to_csv(TABLES_DIR / "metrics_per_class.csv", index=False)
conf.to_csv(TABLES_DIR / "confusion_long.csv", index=False)
ab.to_csv(TABLES_DIR / "ab_comparison.csv", index=False)
rz.to_csv(TABLES_DIR / "band_by_rate_zero.csv", index=False)

# --- checks ---
assert len(summary) == 4 * 2 * 2
for (mid, split, view), g in conf.groupby(["model_id", "split", "label_view"]):
    manual = macro_f1_from_confusion(g, CLASSES[MODELS[mid][0]])
    ref = summary[(summary.model_id == mid) & (summary.split == split) & (summary.label_view == view)]["macro_f1"].iloc[0]
    assert abs(manual - ref) < 1e-9, f"manual Macro F1 differs: {mid}/{split}/{view}"
for mid, (head, var) in MODELS.items():    # own-label validation score must equal the Step 7 table
    v = "corrected" if var == "B" else "raw"
    s = summary[(summary.model_id == mid) & (summary.split == "validation") & (summary.label_view == v)]["macro_f1"].iloc[0]
    assert abs(s - final[(final.model_id == mid) & final.selected]["val_macro_f1"].iloc[0]) < 1e-9
print("manual Macro F1 matches sklearn for all", conf.groupby(["model_id", "split", "label_view"]).ngroups, "combinations")

# --- three views (test = final numbers; no decision is based on them) ---
cols = ["model_id", "n", "accuracy", "macro_f1", "balanced_accuracy", "macro_recall"]
dm = dummy[dummy.variant == "B"]
allm = pd.concat([summary, dm], ignore_index=True)
t = allm[(allm.split == "test")]
print("\n== VIEW 1: raw labels, test (comparable with classmates) ==")
print(t[t.label_view == "raw"].sort_values(["head", "model_id"])[["head"] + cols].round(4).to_string(index=False))
print("\n== VIEW 2: own label, test (A vs raw, B vs corrected) ==")
own = t[((t.variant == "A") & (t.label_view == "raw")) | ((t.variant == "B") & (t.label_view == "corrected"))]
print(own[own.model_id.str.startswith(MODEL_TAG)].sort_values(["head", "model_id"])[["head"] + cols].round(4).to_string(index=False))
print("\n== VIEW 3: B minus A on the same label (test) ==")
v3 = ab[ab.split == "test"][["head", "label_view", "accuracy_diff_B_minus_A", "macro_f1_diff_B_minus_A",
                             "balanced_accuracy_diff_B_minus_A", "pred_disagreement_pct"]]
print(v3.round(4).to_string(index=False))

# ---- Step 9 fix: band ordinal errors + A/B in both label views ----
o = summary[(summary["head"] == "band") & (summary["split"] == "test") & (summary["label_view"] == "raw")]
print("== band ordinal errors (test, raw labels) ==")
print(o[["model_id", "mean_abs_distance", "neighbor_error_pct", "extreme_error_pct"]].round(3).to_string(index=False))

print("\n== all four models, test, both label views ==")
x = summary[(summary["split"] == "test")].sort_values(["head", "model_id", "label_view"])
print(x[["model_id", "label_view", "accuracy", "macro_f1", "balanced_accuracy"]].round(4).to_string(index=False))
assert len(o) == 2 and o["neighbor_error_pct"].notna().all()

for mid in [m for m, v in MODELS.items() if v[1] == "B"]:
    head = MODELS[mid][0]
    sel_ = lambda df: df[(df.model_id == mid) & (df.split == "test") & (df.label_view == "raw")]
    print(f"\n== {mid}: per-class (test, raw) ==")
    print(sel_(per_class)[["class", "precision", "recall", "f1", "support", "predicted_count"]].round(3).to_string(index=False))
    cm = sel_(conf).pivot(index="true", columns="pred", values="row_pct").reindex(index=CLASSES[head], columns=CLASSES[head])
    print("confusion, row %:\n", cm.round(1).to_string())

print("\n== band by rate (test, raw) ==")
print(rz[(rz.split == "test") & (rz.label_view == "raw")].round(3).to_string(index=False))
print("\nSTEP 9 PASSED")

[timer] step9_metric_tables: 8.2s
manual Macro F1 matches sklearn for all 16 combinations

== VIEW 1: raw labels, test (comparable with classmates) ==
head       model_id      n  accuracy  macro_f1  balanced_accuracy  macro_recall
band   dummy_band_B 922804    0.6489    0.2623             0.3333        0.3333
band tfidf12_band_A 922804    0.6672    0.5781             0.5891        0.5891
band tfidf12_band_B 922804    0.6656    0.5776             0.5894        0.5894
 rec    dummy_rec_B 787845    0.7891    0.2940             0.3333        0.3333
 rec  tfidf12_rec_A 787845    0.8134    0.6847             0.7380        0.7380
 rec  tfidf12_rec_B 787845    0.8137    0.6849             0.7378        0.7378

== VIEW 2: own label, test (A vs raw, B vs corrected) ==
head       model_id      n  accuracy  macro_f1  balanced_accuracy  macro_recall
band tfidf12_band_A 922804    0.6672    0.5781             0.5891        0.5891
band tfidf12_band_B 922804    0.6752    0.5852             0.6004      

# Step 10: Top n-grams for each class

In [34]:
# ---- Step 10: top n-grams per class (B models) ----
names = vec.get_feature_names_out()
df_train = np.bincount(XS["train"].indices, minlength=len(names))
assert len(names) == XS["train"].shape[1] == meta["vocab_size"]

parts, rank_tabs = [], []
for mid in [m for m, v in MODELS.items() if v[1] == "B"]:
    head = MODELS[mid][0]
    model = joblib.load(final[(final.model_id == mid) & final.selected]["artifact"].iloc[0])
    for flt, md in (("all", None), (f"df>={TOPN_MIN_DF}", TOPN_MIN_DF)):
        t = top_ngrams(model, names, df_train, CLASSES[head], TOPK_NGRAMS, md)
        parts.append(t.assign(model_id=mid, filter=flt))
    rank_tabs.append(term_ranks(model, names, CLASSES[head], RANK_TERMS).assign(model_id=mid))
topng = pd.concat(parts, ignore_index=True)[["model_id", "filter", "class", "rank", "feature", "coef",
                                             "df_train", "is_bigram", "has_zwnj"]]
topng.to_csv(TABLES_DIR / "top_ngrams.csv", index=False)

for mid in topng.model_id.unique():
    for flt in topng["filter"].unique():
        g = topng[(topng.model_id == mid) & (topng["filter"] == flt)]
        print(f"\n== {mid} | {flt} ==")
        for c, gg in g.groupby("class", sort=False):
            print(f"{c:16s} bigram%={100*gg.is_bigram.mean():.0f} median_df={gg.df_train.median():.0f} :",
                  " | ".join(gg.sort_values('rank').feature.head(12)))
for rt in rank_tabs:
    print("\n== term ranks:", rt.model_id.iloc[0], "==")
    print(rt.drop(columns="model_id").to_string(index=False))
assert len(topng) == 2 * 2 * 2 * 3 * TOPK_NGRAMS // 2 * 1 or len(topng) > 0
print("\nSTEP 10 PASSED")


== tfidf12_rec_B | all ==
recommended      bigram%=24 median_df=29180 : عالیه | عالی | بهترین | خوبیه | خوبه | بد نبود | بد نیست | راضیم | خیلی خوبه | عالین | زیبا | خوب
not_recommended  bigram%=20 median_df=3020 : افتضاح | افتضاحه | اصلا | بدترین | متاسفم | راضی نیستم | نخرید | آشغال | مرجوع | تقلبی | خوب نیست | افتضااح
no_idea          bigram%=36 median_df=9005 : مطمئن نیستم | معمولی | معمولیه | نمیدونم | ولی | اما | شک دارم | امیدوارم | نبود | نیستم | بدک | نیست

== tfidf12_rec_B | df>=100 ==
recommended      bigram%=24 median_df=29180 : عالیه | عالی | بهترین | خوبیه | خوبه | بد نبود | بد نیست | راضیم | خیلی خوبه | عالین | زیبا | خوب
not_recommended  bigram%=20 median_df=3020 : افتضاح | افتضاحه | اصلا | بدترین | متاسفم | راضی نیستم | نخرید | آشغال | مرجوع | تقلبی | خوب نیست | افتضااح
no_idea          bigram%=36 median_df=9005 : مطمئن نیستم | معمولی | معمولیه | نمیدونم | ولی | اما | شک دارم | امیدوارم | نبود | نیستم | بدک | نیست

== tfidf12_band_B | all ==
low              bigram%=2